# 4D SfM Pipeline

Processes one new day at a time through the full 4D SfM workflow. All logic
lives in `tlapse4d.pipeline_4dsfm.run_4dsfm_day`; this notebook is just config +
one call.

Steps run by `run_4dsfm_day` (each skips if its output already exists):

1. Multi-temporal bundle adjustment — reference cameras pinned tight (0.001 m),
   new-day cameras loose; new-day IOP floating, ref IOP fixed per-day.
2. Single-day re-run with **fixed** IOP from Step 1.
3. ASP three-stage ICP (`point-to-plane` → `similarity-p2p` → stable terrain only).
4. Apply ASP transform to the single-day camera EOPs.
5. _(no-op — verification only)_
6. Rebuild the cloud from the corrected Metashape chunk transform (same session
   as the matrix fix, so it isn't lost on save/reload).
7. Append the validated day to the reference registry.

Step 3b runs M3C2 on the coreg cloud; Step 6b validates that the rebuilt cloud
matches the coreg cloud (median ≈ 0 m → transform propagated correctly).


In [1]:
%load_ext autoreload
%autoreload 2

import os
os.environ["AGISOFT_LICENSE_PATH"] = "/home/asus/.config/Agisoft/license.lic"

from pathlib import Path

import Metashape  # noqa: F401  — must import after AGISOFT_LICENSE_PATH is set
from tlapse4d.pipeline_4dsfm import run_4dsfm_day


## Configuration

Set all paths and parameters here. Defaults match the values that produced the
validated 2023-12-15 run; tweak per-day if needed.


In [2]:
# ── Site — edit the 3 paths in site_config.py ────────────────────────────
import site_config as site

# ── Date to process ──────────────────────────────────────────────────────
new_date = "2023-12-15"

# ── Pipeline parameters (defaults shown — override as needed) ────────────
params = dict(
    match_downscale = 1,
    depth_downscale = 2,
    loc_acc_new     = (0.5, 0.5, 0.5),
    rot_acc_new     = (5.0, 5.0, 5.0),
    ref_downsample  = 0.4,
    tba_downsample  = 1.0,
    p2p_max_disp    = 10.0,
    sp2p_max_disp   =  5.0,
    m_sp2p_max_disp =  2,
    use_ecef        = True,
    overwrite       = False,   # True forces full recompute
    verbose         = True,    # print pc_align stdout
)

## Run

`overwrite=False` means each step skips itself if its key output exists on
disk — useful for resuming after a crash. Set `overwrite=True` above for a
fresh run.


In [ ]:
result = run_4dsfm_day(
    new_date     = new_date,
    tlcam_dir    = site.tlcam_dir,
    ref_cloud    = site.ref_cloud,
    glacier_mask = site.glacier_mask,
    registry_csv = site.registry_csv,
    output_dir   = site.output_dir,
    **params,
)

print()
print(f"Coreg M3C2  : before {result['coreg_med_before']:+.4f} m  →  after {result['coreg_med_after']:+.4f} m")
print(f"Validation  : median {result['validation_med']:+.4f} m  std {result['validation_std']:.4f} m")
print(f"Validated   : {result['validated_laz']}")